# Looking at sentiment of articles of sectoral leaders

In [1]:
import pandas as pd

reuters_data = pd.read_csv('../data/reuters/combined.csv')
reuters_data = reuters_data[['title','description','date']]
reuters_data

,title,description,date
0,Insurer Hartford to pay $650 million for claim...,Insurer Hartford Financial Services Group said...,2021-04-16
1,Congress poised for battle over reparations fo...,Congress poised for battle over reparations fo...,2021-04-16
2,"Knighthead, Certares sweeten bid to fund Hertz...",Investment firms Knighthead Capital Management...,2021-04-16
3,Google misled consumers over data collection -...,SYDNEY (Reuters) -Australia's federal court fo...,2021-04-16
4,"With flagship electric car, Mercedes takes fig...","Daimler AG on Thursday unveiled the electric ""...",2021-04-16
...,...,...,...
49672,Cambridge Analytica played key Trump campaign ...,The suspended chief executive of UK-based poli...,2018-03-20
49673,"Factbox: How United States, others regulate au...",An Uber self-driving sport utility vehicle str...,2018-03-20
49674,Senate Democrat wants Facebook CEO Zuckerberg ...,"U.S. Senator Dianne Feinstein, the top Democra...",2018-03-20
49675,Factbox: Who is Cambridge Analytica and what d...,British data analytics firm Cambridge Analytic...,2018-03-20


Five industries in the S&P500 have a weight of over 10% in the Index:
* Information Technology 26.7%
* Health Care 12.8%
* Consumer Discretionary 12.7%
* Financials 11.5%
* Communication Services 11.2%


For each of this the largest company in each industry (May 2021)
* Information Technology: 
    * Apple Inc. APPL
* Health Care: 
    * Johnson & Johnson JNJ
* Consumer Discretionary
    * Amazon.com Inc AMZN
* Financials 11.5% 
    * Berkshire Hathaway BRK.B. Will disregard because of it being a conglomerate and go with JP. Morgan JPM instead
* Communication Services 11.2%
    * Facebook Inc A FB. Personally I would go for AT&T but I believe Facebook has received a large amount of news and is therefore interesting to look at


In [2]:
# Keywords to filter articles

appl_keywords = ['Apple', 'Tim Cook', 'iOS', 'iPhone', 'App Store', 'APPL']
jnj_keywords = ['Johnson & Johnson', 'Johnson and Johnson', 'J&J', 'Jansen', 'Alex Gorsky']
amzn_keywords = ['Amazon', 'Jeff Bezos', 'AWS', 'Twitch', 'Audible', 'Alexa']
jpm_keywords = ['JPMorgan Chase', 'J.P. Morgan', 'JP Morgan','JPM', 'Jamie Dimon']
fb_keywords = ['Facebook', 'Instagram', 'Mark Zuckerberg', 'FB']

In [18]:
def get_stories_with_keywords(keywords, articles=reuters_data):
    keyword_stories = []
    for index, row in articles.iterrows():
        headline = row['title']
        description = row['description']
        if any(keyword in headline for keyword in keywords) or any(keyword in description for keyword in keywords):
            keyword_stories.append(articles.iloc[index])
        # if(index % 999 == 0):
          #  print('Processed {} stories, found {} related stories.'.format(index + 1, len(keyword_stories)))
    
    return keyword_stories

In [17]:
appl_stories = get_stories_with_keywords(appl_keywords)

Processed 1 stories, found 0 related stories.
Processed 1000 stories, found 16 related stories.
Processed 1999 stories, found 24 related stories.
Processed 2998 stories, found 33 related stories.
Processed 3997 stories, found 49 related stories.
Processed 4996 stories, found 71 related stories.
Processed 5995 stories, found 83 related stories.
Processed 6994 stories, found 106 related stories.
Processed 7993 stories, found 121 related stories.
Processed 8992 stories, found 138 related stories.
Processed 9991 stories, found 159 related stories.
Processed 10990 stories, found 178 related stories.
Processed 11989 stories, found 196 related stories.
Processed 12988 stories, found 229 related stories.
Processed 13987 stories, found 250 related stories.
Processed 14986 stories, found 274 related stories.
Processed 15985 stories, found 286 related stories.
Processed 16984 stories, found 295 related stories.
Processed 17983 stories, found 299 related stories.
Processed 18982 stories, found 304

In [19]:
jnj_stories = get_stories_with_keywords(jnj_keywords)
amzn_stories = get_stories_with_keywords(amzn_keywords)
jpm_stories = get_stories_with_keywords(jpm_keywords)
fb_stories = get_stories_with_keywords(fb_keywords)

In [181]:
import nltk
from nltk.sentiment import vader

analyser = vader.SentimentIntensityAnalyzer()

def find_sentiments(company_stories_list):
    company_stories = pd.DataFrame(company_stories_list)
    company_stories.reset_index(inplace=True)
    
    headline_sentiment = company_stories['title'].apply(analyser.polarity_scores)
    description_sentiment = company_stories['description'].apply(analyser.polarity_scores)

    company_stories = company_stories.join(pd.DataFrame(headline_sentiment.to_list()), how='right')
    company_stories = company_stories.join(pd.DataFrame(headline_sentiment.to_list()), how='right', rsuffix='_description')

    # Clusters the stories according to the date
    dates = company_stories['date'].drop_duplicates().values
    mean_daily_sentiment = [company_stories.loc[company_stories['date'] == date].mean() for date in dates]
    
    daily_sentiments = pd.DataFrame(dates)
    daily_sentiments = daily_sentiments.join(pd.DataFrame(mean_daily_sentiment), how='right')

    daily_sentiments['combined compound'] = (daily_sentiments['compound'] + daily_sentiments['compound_description']) / 2
    daily_sentiments['combined negative'] = (daily_sentiments['neg'] + daily_sentiments['neg_description']) / 2
    daily_sentiments['combined positive'] = (daily_sentiments['pos'] + daily_sentiments['pos_description']) / 2
    
    daily_sentiments.set_index(0, inplace=True)
    daily_sentiments = daily_sentiments[['combined compound', 'combined negative', 'combined positive']]
    
    #Filtering out the days where the mean is 0
    daily_sentiments = daily_sentiments[daily_sentiments['combined compound'] != 0]
    
    return daily_sentiments

# Investigating Apple
## Sentiment

In [151]:
apple_sentiment = find_sentiments(appl_stories)
apple_sentiment

,combined compound,combined negative,combined positive
0,,,
2021-04-15,0.1366,0.000,0.087
2021-04-14,-0.1738,0.187,0.000
2021-04-12,0.3612,0.000,0.200
2021-04-11,0.2023,0.000,0.141
2021-04-09,-0.3818,0.224,0.000
...,...,...,...
2018-04-19,-0.3400,0.255,0.000
2018-04-03,-0.1027,0.167,0.000
2018-03-30,0.4404,0.000,0.209


## Returns

In [85]:
from iexfinance.stocks import Stock
aapl = Stock('AAPL', output_format='pandas', token=open('../data/secret.txt', "r").read())

In [88]:
# Historical data endpoint is not related to the Stock
from iexfinance.stocks import get_historical_data

aapl_prices = get_historical_data('AAPL', start='2018-02-22', end='2021-04-16', close_only=True, token=open('../data/secret.txt', "r").read())
aapl_prices

,close,volume
2018-02-22,43.125,123967760
2018-02-23,43.875,135249440
2018-02-26,44.7425,152648696
2018-02-27,44.5975,155712500
2018-02-28,44.53,151128552
...,...,...
2021-04-12,131.24,91419983
2021-04-13,134.43,91266545
2021-04-14,132.03,87222782
2021-04-15,134.5,89347102


In [184]:
# Careful, uses credits. Personal API stored in secret.txt
def get_closing_prices(ticker):
    prices = get_historical_data(ticker, start='2018-02-22', end='2021-04-16', close_only=True, token=open('../data/secret.txt', "r").read())
    prices.to_csv('../data/companies/{}.csv'.format(ticker))
    return prices

In [99]:
aapl_prices = aapl_prices.iloc[::-1]

In [94]:
aapl_prices.to_csv('../data/companies/aapl.csv')

In [104]:
import numpy as np
aapl_change = aapl_prices.copy()
aapl_change = aapl_change.iloc[::-1]

day_change = []
day_change.append(np.nan) # Make sure the first row is filled

for i in range (1, len(aapl_change)):
    previous_close = aapl_change.iloc[int(i) - 1]['close']
    todays_close = aapl_change.iloc[i]['close']
    day_change.append(1 + ((todays_close - previous_close)/previous_close))
    
aapl_change['change'] = day_change
aapl_change

,close,volume,change
2018-02-22,43.125,123967760,NaN
2018-02-23,43.875,135249440,1.017391
2018-02-26,44.7425,152648696,1.019772
2018-02-27,44.5975,155712500,0.996759
2018-02-28,44.53,151128552,0.998486
...,...,...,...
2021-04-12,131.24,91419983,0.986804
2021-04-13,134.43,91266545,1.024307
2021-04-14,132.03,87222782,0.982147
2021-04-15,134.5,89347102,1.018708


In [217]:
# For each date, calculates the returns in the next 1, 3, 5, and 10 trading days (cumulative return and increase True/False)
def calculate_future_returns(df):
    for num_days in [1,3,5,10]:
        column_name = 'Future {}D'.format(num_days)
        df[column_name] = np.nan
        
        for i in range(0, len(df) - num_days):
            # Takes the next num_days and takes the value of the cumulative product of the last day
            if(num_days == 1):
                result = df['change'].iloc[i + 1]
            else:
                result = df['change'].iloc[i + 1: i + 1 + num_days].cumprod()[-1]

            df[column_name].iloc[i] = result

        df[column_name + ' increase'] = df[column_name] > 1.0
        
    return df

In [210]:
aapl_change = calculate_future_returns(aapl_change)
aapl_change.tail(11)

/Users/oenmalm/anaconda3/lib/python3.8/site-packages/pandas/core/indexing.py:671: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_with_indexer(indexer, value)


,close,volume,change,Future 1D,Future 1D increase,Future 3D,Future 3D increase,Future 5D,Future 5D increase,Future 10D,Future 10D increase
2021-04-01,123,75089134,1.006959,1.023577,True,1.039837,True,1.081260,True,1.090732,True
2021-04-05,125.9,88651175,1.023577,1.002462,True,1.035425,True,1.042415,True,NaN,False
2021-04-06,126.21,80171253,1.002462,1.013390,True,1.053760,True,1.065130,True,NaN,False
2021-04-07,127.9,83466716,1.013390,1.019234,True,1.026114,True,1.032291,True,NaN,False
2021-04-08,130.36,88844591,1.019234,1.020213,True,1.031221,True,1.031758,True,NaN,False
2021-04-09,132.995,106686703,1.020213,0.986804,False,0.992744,False,1.008760,True,NaN,False
2021-04-12,131.24,91419983,0.986804,1.024307,True,1.024840,True,NaN,False,NaN,False
2021-04-13,134.43,91266545,1.024307,0.982147,False,0.997992,False,NaN,False,NaN,False
2021-04-14,132.03,87222782,0.982147,1.018708,True,NaN,False,NaN,False,NaN,False
2021-04-15,134.5,89347102,1.018708,0.997472,False,NaN,False,NaN,False,NaN,False


In [152]:
aapl_sentiments_returns = apple_sentiment.join(aapl_change, how='inner')

In [153]:
aapl_sentiments_returns.tail(5)

,combined compound,combined negative,combined positive,close,volume,change,Future 1D,Future 1D increase,Future 3D,Future 3D increase,Future 5D,Future 5D increase,Future 10D,Future 10D increase
2021-04-08,-0.30405,0.176,0.000,130.36,88844591,1.019234,1.020213,True,1.031221,True,1.031758,True,NaN,False
2021-04-09,-0.38180,0.224,0.000,132.995,106686703,1.020213,0.986804,False,0.992744,False,1.008760,True,NaN,False
2021-04-12,0.36120,0.000,0.200,131.24,91419983,0.986804,1.024307,True,1.024840,True,NaN,False,NaN,False
2021-04-14,-0.17380,0.187,0.000,132.03,87222782,0.982147,1.018708,True,NaN,False,NaN,False,NaN,False
2021-04-15,0.13660,0.000,0.087,134.5,89347102,1.018708,0.997472,False,NaN,False,NaN,False,NaN,False


## Generating model

In [154]:
from sklearn.naive_bayes import GaussianNB
gnb = GaussianNB()

In [211]:
training_1D_X = pd.concat([aapl_sentiments_returns[:'2020-01-01'], aapl_sentiments_returns['2020-12-31':'2021-04-01']])['combined compound']
training_1D_y = pd.concat([aapl_sentiments_returns[:'2020-01-01'], aapl_sentiments_returns['2020-12-31':'2021-04-01']])['Future 1D increase']
# training_1D_y = pd.concat([aapl_sentiments_returns[:'2020-01-01'], aapl_sentiments_returns['2020-12-31':]])[['Future 1D increase','Future 3D increase','Future 5D increase', 'Future 10D increase']]

test_1D_X = aapl_sentiments_returns['2020-01-01':'2020-12-31']['combined compound']
test_1D_y = aapl_sentiments_returns['2020-01-01':'2020-12-31']['Future 1D increase']
# test_1D_y = aapl_sentiments_returns['2020-01-01':'2020-12-31'][['Future 1D increase','Future 3D increase','Future 5D increase', 'Future 10D increase']]

In [212]:
gnb_prediction = gnb.fit(training_1D_X.values.reshape(-1, 1), training_1D_y.values.reshape(-1, 1)).predict(test_1D_X.values.reshape(-1, 1))
print("Number of mislabeled points out of a total %d points : %d" % (len(test_1D_X), (gnb_predicition != test_1D_y).sum()))

Number of mislabeled points out of a total 106 points : 43


/Users/oenmalm/anaconda3/lib/python3.8/site-packages/sklearn/utils/validation.py:73: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  return f(**kwargs)


In [303]:
type_1_error = 0 # (false positive)
type_2_error = 0 # (false negative)
for i in range(0,len(gnb_prediction)):
    if gnb_prediction[i]:
        if not test_1D_y[i]:
            type_1_error += 1
    else:
        if test_1D_y[i]:
            type_2_error += 1

print('Type 1 Error: {} ({})'.format(type_1_error, type_1_error / len(gnb_prediction)))
print('Type 2 Error: {} ({})'.format(type_2_error, type_2_error / len(gnb_prediction)))

Type 1 Error: 36 (0.33962264150943394)
Type 2 Error: 7 (0.0660377358490566)


In [166]:
gnb_predicition

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True, False,  True,  True, False,
        True, False,  True,  True,  True, False,  True,  True,  True,
       False,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True, False,  True, False,  True,  True, False, False,  True,
        True,  True,  True,  True, False,  True,  True,  True, False,
        True,  True, False,  True,  True,  True,  True,  True,  True,
       False,  True,  True,  True,  True,  True,  True, False,  True,
        True,  True,  True,  True,  True, False,  True])

It looks a great deal better than for the index. Lets do this for every stock that I wanted to cover...

# Looking at the remaining companies
## Sentiments

In [191]:
# Stories with the keywords and the sentiment of company-related stories for the individual days

jnj_stories = get_stories_with_keywords(jnj_keywords)
amzn_stories = get_stories_with_keywords(amzn_keywords)
jpm_stories = get_stories_with_keywords(jpm_keywords)
fb_stories = get_stories_with_keywords(fb_keywords)

aapl_sentiment = apple_sentiment

jnj_sentiment = find_sentiments(jnj_stories)
amzn_sentiment = find_sentiments(amzn_stories)
jpm_sentiment = find_sentiments(jpm_stories)
fb_sentiment = find_sentiments(fb_stories)

## Returns

In [200]:
# aapl_change = calculate_future_returns(aapl_change)
def format_returns(prices):
    '''Converts daily price data into a dataframe containing information of the next 1,3,5, and 10 day returns'''
    daily_changes = calculate_daily_change(prices)
    future_changes = calculate_future_returns(daily_changes)
    return future_changes

In [215]:
def calculate_daily_change(ticker_prices):
    '''Takes the prices from IEX and calculates the daily change as 1 + r'''
    ticker_change = ticker_prices.copy()
    
    #Sorts the date accending
    ticker_change = ticker_change.iloc[::-1]

    daily_change = []
    daily_change.append(np.nan) # Make sure the first row is filled (no previous day to calculate with)

    for i in range (1, len(aapl_change)):
        previous_close = ticker_change['close'].iloc[int(i) - 1]
        todays_close = ticker_change['close'].iloc[i]
        daily_change.append(1 + ((todays_close - previous_close)/previous_close))

    ticker_change['change'] = daily_change
    
    return ticker_change

In [193]:
news_overview = pd.DataFrame(index=['Apple', 'Johnson & Johnson', 'Amazon', 'J.P. Morgan', 'Facebook'])
news_overview['article count'] = [len(company_stories) for company_stories in [appl_stories, jnj_stories, amzn_stories, jpm_stories, fb_stories]]
news_overview['days non-0 compound sentiment'] = [len(daily_company_sentiments) for daily_company_sentiments in [apple_sentiment, jnj_sentiment, amzn_sentiment, jpm_sentiment, fb_sentiment]]

news_overview

,article count,days non-0 compound sentiment
Apple,909,356
Johnson & Johnson,152,101
Amazon,988,521
J.P. Morgan,340,180
Facebook,1090,380


In [194]:
len(reuters_data)

49677

In [199]:
jnj_prices = get_closing_prices('JNJ')
amzn_prices = get_closing_prices('AMZN')
jpm_prices = get_closing_prices('JPM')
fb_prices = get_closing_prices('FB')

In [218]:
jnj_returns = format_returns(jnj_prices)
amzn_returns = format_returns(amzn_prices)
jpm_returns = format_returns(jpm_prices)
fb_returns = format_returns(fb_prices)

/Users/oenmalm/anaconda3/lib/python3.8/site-packages/pandas/core/indexing.py:671: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_with_indexer(indexer, value)


In [295]:
companies = ['Apple', 'Johnson & Johnson', 'Amazon', 'J.P. Morgan', 'Facebook']
training_metrics = ['training cases','test cases', 'correct', 'test positive returns','test non-positive returns','false positives', 'false negatives']
company_training_overview = pd.DataFrame(index=companies, columns=training_metrics)

In [310]:
add_model_details_to_df('Apple', aapl_change, apple_sentiment, company_training_overview)
company_training_overview

/Users/oenmalm/anaconda3/lib/python3.8/site-packages/sklearn/utils/validation.py:73: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  return f(**kwargs)


,training cases,test cases,correct,test positive returns,test non-positive returns,false positives,false negatives
Apple,221,106,63 (59.43%),62 (58.49%),44 (41.51%),36.0 (33.96%),7.0 (6.60%)
Johnson & Johnson,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Amazon,NaN,NaN,NaN,NaN,NaN,NaN,NaN
J.P. Morgan,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Facebook,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [315]:
add_model_details_to_df('Johnson & Johnson', jnj_returns, jnj_sentiment)
add_model_details_to_df('Amazon', amzn_returns, amzn_sentiment)
add_model_details_to_df('J.P. Morgan', jpm_returns, jpm_sentiment)
add_model_details_to_df('Facebook', fb_returns, fb_sentiment)
company_training_overview

/Users/oenmalm/anaconda3/lib/python3.8/site-packages/sklearn/utils/validation.py:73: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  return f(**kwargs)
/Users/oenmalm/anaconda3/lib/python3.8/site-packages/sklearn/utils/validation.py:73: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  return f(**kwargs)
/Users/oenmalm/anaconda3/lib/python3.8/site-packages/sklearn/utils/validation.py:73: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  return f(**kwargs)
/Users/oenmalm/anaconda3/lib/python3.8/site-packages/sklearn/utils/validation.py:73: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), 

,training cases,test cases,correct,test positive returns,test non-positive returns,false positives,false negatives
Apple,221,106,63 (59.43%),62 (58.49%),44 (41.51%),36.0 (33.96%),7.0 (6.60%)
Johnson & Johnson,67,29,16 (55.17%),12 (41.38%),17 (58.62%),12.0 (41.38%),1.0 (3.45%)
Amazon,286,165,92 (55.76%),73 (44.24%),92 (55.76%),0.0 (0.00%),73.0 (44.24%)
J.P. Morgan,98,70,29 (41.43%),41 (58.57%),29 (41.43%),0.0 (0.00%),41.0 (58.57%)
Facebook,198,141,66 (46.81%),63 (44.68%),78 (55.32%),45.0 (31.91%),30.0 (21.28%)


In [314]:
def add_model_details_to_df(index_company, future_returns, sentiment_data, df=company_training_overview):
    predictive_metrics = get_predictive_metrics(future_returns,sentiment_data)
    for training_metric in training_metrics:
        df[training_metric].loc[index_company] = predictive_metrics[training_metric]
    
    return df

In [311]:
def get_predictive_metrics(future_returns, sentiment_data):
    results = {
        'training cases': None, 
        'test cases':  None,
        'correct': None,
        'test positive returns': None,
        'test non-positive returns': None,
        'false positives': None,
        'false negatives': None
    }
    
    # All of the lists apart from apple need to be flipped, comment .iloc[::-1] when doing Apple
    sentiments_returns = sentiment_data.join(future_returns, how='inner').iloc[::-1]
    training_1D_X = pd.concat([sentiments_returns[:'2020-01-01'], sentiments_returns['2020-12-31':'2021-04-01']])['combined compound']
    assert(len(training_1D_X)) > 10
    
    training_1D_y = pd.concat([sentiments_returns[:'2020-01-01'], sentiments_returns['2020-12-31':'2021-04-01']])['Future 1D increase']
    assert(len(training_1D_y)) > 10
    results['training cases'] = len(training_1D_X)

    test_1D_X = sentiments_returns['2020-01-01':'2020-12-31']['combined compound']
    assert(len(test_1D_X) > 10)
    test_1D_y = sentiments_returns['2020-01-01':'2020-12-31']['Future 1D increase']
    assert(len(test_1D_y) > 10)
    results['test cases'] = len(test_1D_X)
    
    
    
    gnb_prediction = gnb.fit(training_1D_X.values.reshape(-1, 1), training_1D_y.values.reshape(-1, 1)).predict(test_1D_X.values.reshape(-1, 1))
    correctly_guessed = (gnb_prediction == test_1D_y).sum()
    results['correct'] = '{} ({:.2f}%)'.format(correctly_guessed, (correctly_guessed / len(gnb_prediction)) * 100)
    
    
    t1_error_count = 0.0 # (false positive)
    t2_error_count = 0.0 # (false negative)
    for i in range(0,len(gnb_prediction)):
        if gnb_prediction[i]:
            if not test_1D_y[i]:
                t1_error_count += 1
        else:
            if test_1D_y[i]:
                t2_error_count += 1
                
    results['false positives'] = '{} ({:.2f}%)'.format(t1_error_count, (t1_error_count / len(gnb_prediction)) * 100)
    results['false negatives'] = '{} ({:.2f}%)'.format(t2_error_count, (t2_error_count / len(gnb_prediction)) * 100)

    positive_returns_in_test = len([positive for positive in test_1D_y if positive==True])
    results['test positive returns'] = '{} ({:.2f}%)'.format(positive_returns_in_test, (positive_returns_in_test / len(test_1D_y)) * 100)
    results['test non-positive returns'] =  '{} ({:.2f}%)'.format(len(test_1D_y) - positive_returns_in_test, ((len(test_1D_y) - positive_returns_in_test)/len(test_1D_y)) * 100)
    

    
    return results